# 06 — Edge-IIoTset: Provisional Static Split + Decision Tree Baseline

**Project:** J26-CS-336 · Network Flow V2  
**CSV:** `ML-EdgeIIoT-dataset.csv` · 157,800 reported observations · `Attack_label` / `Attack_type`

**Read first:** The downloaded selected ML CSV is *not* verified for temporal windows. Its `frame.time` column has label-associated formats and is excluded. This notebook trains **only a provisional static binary classifier** and evaluates **validation** results; test labels are preserved for a future final, locked evaluation.

The resulting score **does not prove** new-capture, device-independent, APT campaign or Raspberry Pi detection. We lack verified capture/session grouping. Same-feature grouping guards against exact input duplicates, *not* broader collection-artifact leakage.

### Method
1. Verify the unchanged CSV via SHA-256 and label consistency.
2. Use a small, explainable *network metadata* feature subset, excluding timestamp, IP, raw payload, ports, labels and raw text.
3. Make a provisional approximately 70/15/15 **group-disjoint** split, grouping identical model-input vectors, and report achieved proportions and all class counts.
4. Fit numeric imputation on **training only**, train one Decision Tree baseline, score on **validation only**.
5. Save a split manifest, local model and validation-only report. **No test evaluation.**

This is a diagnostic baseline and may be stopped by automated integrity gates.

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import hashlib, json, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, average_precision_score, roc_auc_score, accuracy_score
import joblib

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
CSV = ROOT / "data/raw/edge_iiotset/ML-EdgeIIoT-dataset.csv"
SPLIT_DIR = ROOT / "data/splits/edge_iiotset"
REPORT_DIR = ROOT / "reports/metrics"
MODEL_DIR = ROOT / "models"
EXPECTED_SHA256 = "53101fad091af20bee815860962a5016b802ee45d456a141042f6183094c3c1a"
SEED = 42
for p in (SPLIT_DIR, REPORT_DIR, MODEL_DIR): p.mkdir(parents=True, exist_ok=True)
assert CSV.exists(), f"File missing: {CSV}"
print("Source file:",CSV)

In [ ]:
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(chunk_size), b''):
            h.update(block)
    return h.hexdigest()

actual_hash=sha256_file(CSV)
print('SHA-256:', actual_hash)
assert actual_hash == EXPECTED_SHA256, 'Dataset changed since Step 01; STOP and investigate, do not reuse previous audits.'
df = pd.read_csv(CSV, low_memory=False)
print('Rows, columns:',df.shape)
assert len(df) == 157800 and df.shape[1] == 63, 'Unexpected dataset shape; STOP.'
assert {'Attack_label','Attack_type'}.issubset(df.columns)
assert set(df['Attack_label'].dropna().unique()) == {0,1}, 'Binary label must be 0/1.'
assert not df[['Attack_label','Attack_type']].isna().any().any()
assert ((df['Attack_label'].eq(0)) == df['Attack_type'].eq('Normal')).all(), 'Label mismatch; STOP.'
print('Label counts:',df['Attack_label'].value_counts().to_dict())

## Feature selection — intentionally conservative

These fields are a **candidate starting point**, not independently verified production-ready flow statistics. They describe packet/protocol metadata. Source IP/port, timestamp, payload, HTTP URI, raw content, and all labels are excluded to reduce obvious shortcuts. The feature set is frozen **before** split/model comparison.

If a field is constant or never numeric in this release, the gate will stop; revise the candidate list deliberately rather than silently changing it.

In [ ]:
FEATURES = [
    'arp.opcode',
    'icmp.seq_le',
    'icmp.checksum',
    'http.content_length',
    'http.response',
    'tcp.connection.fin',
    'tcp.connection.rst',
    'tcp.connection.syn',
    'tcp.connection.synack',
    'tcp.flags.ack',
    'tcp.len',
    'udp.time_delta',
    'dns.retransmission',
    'mqtt.len',
    'mqtt.msgtype',
]
assert len(FEATURES) == len(set(FEATURES))
missing=[f for f in FEATURES if f not in df]
assert not missing, f'Missing columns: {missing}'
assert not any(f in {'Attack_label','Attack_type','frame.time','ip.src_host','ip.dst_host'} for f in FEATURES)

X = df[FEATURES].apply(pd.to_numeric, errors='coerce')
parse_rates = (100*X.notna().mean()).round(2)
variability = X.nunique(dropna=True)
feature_audit=pd.DataFrame({'numeric_parse_pct':parse_rates,'distinct_numeric_values':variability,'missing_after_coercion':X.isna().sum()})
display(feature_audit)
assert (parse_rates >= 98).all(), 'One or more chosen features are nonnumeric in >2% of rows. Investigate.'
constant=variability[variability<=1].index.tolist()
if constant:
    print('Training-wide constant inputs (recorded; will be dropped):',constant)
    FEATURES=[f for f in FEATURES if f not in constant]
    X=X[FEATURES]
assert len(FEATURES)>=5,'Too few usable features. STOP.'
print('Frozen baseline features:',FEATURES)

## Duplicate-aware grouping and a provisional holdout

**Group:** identical values across the chosen input features, excluding all ground-truth labels. Conflicting labels within a group are **reported**, not changed. Each group is assigned to one partition only.

A 70/15/15 ratio is a *target*: groups are indivisible, so the achieved row percentages can differ. We try deterministic group splits using a fixed seed sequence and select the split closest to the target **using only label frequencies for stratification**, not model performance. If group sizes/class coverage prevent a meaningful split, the notebook stops and you should report that finding. Original capture/session groups remain unknown.

In [ ]:
# Stable fingerprints of selected-feature vectors. Missing numeric values use an explicit sentinel.
# Pandas hash is used for partition assignment, not as a prediction feature.
Xgroup=X.fillna(-987654321.125)
groups=pd.util.hash_pandas_object(Xgroup, index=False).astype('uint64').to_numpy()
y=df['Attack_label'].astype('int8').to_numpy()
type_y=df['Attack_type'].astype('string')

group_df=pd.DataFrame({'group':groups, 'binary':y, 'attack_type':type_y})
size_per_group=group_df.groupby('group').size()
label_conflict=group_df.groupby('group')['binary'].nunique()
type_conflict=group_df.groupby('group')['attack_type'].nunique()
print('Unique feature groups:',len(size_per_group))
print('Rows in groups of size >1:',int(size_per_group[size_per_group>1].sum()))
print('Largest input group:',int(size_per_group.max()))
print('Groups with conflicting binary labels:',int((label_conflict>1).sum()))
print('Groups with conflicting attack types:',int((type_conflict>1).sum()))
assert size_per_group.max() < len(df)*0.15, 'One feature group is larger than 15% of dataset; 70/15/15 not viable without reviewing feature set.'

In [ ]:
# Candidate splits: 15% of GROUPS for test, then approximately 15/85 of remaining GROUPS for validation.
# Ranking considers row proportion and binary-class distribution (stratification), never model performance.
base_pos = y.mean()
all_classes=set(type_y.dropna().unique())

def score_split(t, v, s):
    counts=np.array([len(t),len(v),len(s)], dtype=float)/len(y)
    size_penalty=np.abs(counts-np.array([0.70,0.15,0.15])).sum()
    balance_penalty=sum(abs(y[ix].mean()-base_pos) for ix in (t,v,s))
    covered=sum(len(all_classes-set(type_y.iloc[ix].unique())) for ix in (t,v,s))
    return 5*size_penalty + balance_penalty + 3*covered

best=None
for attempt in range(20):
    first=GroupShuffleSplit(n_splits=1,test_size=0.15,random_state=SEED+attempt)
    dev_idx,test_idx=next(first.split(X, y, groups=groups))
    second=GroupShuffleSplit(n_splits=1,test_size=0.15/0.85,random_state=SEED+100+attempt)
    train_rel,val_rel=next(second.split(X.iloc[dev_idx],y[dev_idx],groups=groups[dev_idx]))
    train_idx, val_idx = dev_idx[train_rel],dev_idx[val_rel]
    if not all(len(idx)>0 for idx in (train_idx,val_idx,test_idx)): continue
    sc=score_split(train_idx,val_idx,test_idx)
    if best is None or sc<best[0]:
        best=(sc,train_idx,val_idx,test_idx,attempt)
assert best is not None, 'Unable to construct partitions.'
_,train_idx,val_idx,test_idx,attempt=best

# Group disjointness proof.
train_g=set(groups[train_idx]); val_g=set(groups[val_idx]); test_g=set(groups[test_idx])
assert not(train_g & val_g or train_g & test_g or val_g & test_g),'STOP: group overlap.'

partition=np.full(len(df),'unset',dtype=object)
partition[train_idx]='train';partition[val_idx]='validation';partition[test_idx]='test'
assert not (partition=='unset').any()

split_summary=[]
for name,ix in (('train',train_idx),('validation',val_idx),('test',test_idx)):
    pct=100*len(ix)/len(df)
    summary={'partition':name,'rows':int(len(ix)),'row_pct':round(pct,3),
             'attack_pct':round(100*y[ix].mean(),3),
             'attack_type_count':int(type_y.iloc[ix].nunique())}
    split_summary.append(summary)
    print(name, 'top label counts:',type_y.iloc[ix].value_counts().head(5).to_dict())
display(pd.DataFrame(split_summary))

# Conservative gate to prevent a misleading '70/15/15' claim.
ratio_error=max(abs(row['row_pct']-target) for row,target in zip(split_summary,[70,15,15]))
print('Max deviation from target (percentage points):',round(ratio_error,3))
if ratio_error > 5:
    raise RuntimeError('Group sizes prevent a sufficiently close 70/15/15 split. Inspect size distribution and choose a documented alternative before training.')
if any(row['attack_type_count'] < len(all_classes) for row in split_summary):
    raise RuntimeError('At least one attack class missing from a partition; revise split method before training.')

manifest=pd.DataFrame({
    'source_row_index':np.arange(len(df),dtype=int),
    'split':partition,
    'feature_group_hash':groups.astype(str),
    'Attack_label':y,
    'Attack_type':type_y
})
manifest_path=SPLIT_DIR/'edge_iiotset_static_split_manifest_v1.csv'
manifest.to_csv(manifest_path,index=False)
print('Saved split manifest:',manifest_path)

## Train-only preprocessing and Decision Tree (validation only)

Fit imputation and the tree on `train` only. No global normalization is necessary for the initial tree baseline. Testing rows are **not** scored here. Compare this model only with models evaluated using the same partition, feature schema and target.

In [ ]:
pipe=Pipeline([
    ('impute',SimpleImputer(strategy='median',keep_empty_features=True)),
    ('model',DecisionTreeClassifier(max_depth=8,min_samples_leaf=20,random_state=SEED))
])
pipe.fit(X.iloc[train_idx],y[train_idx])
validation_pred=pipe.predict(X.iloc[val_idx])
validation_proba=pipe.predict_proba(X.iloc[val_idx])[:,list(pipe.classes_).index(1)]
cm=confusion_matrix(y[val_idx],validation_pred,labels=[0,1])
tn,fp,fn,tp=[int(q) for q in cm.ravel()]
precision, recall, f1,_=precision_recall_fscore_support(y[val_idx],validation_pred,average='binary',zero_division=0)
metrics={
 'accuracy':float(accuracy_score(y[val_idx],validation_pred)),
 'precision_attack':float(precision),'recall_attack':float(recall),'f1_attack':float(f1),
 'false_positive_rate':float(fp/(fp+tn)) if (fp+tn)>0 else None,
 'false_alerts_per_100k_normal':float(100000*fp/(fp+tn)) if (fp+tn)>0 else None,
 'average_precision_attack':float(average_precision_score(y[val_idx],validation_proba)),
 'roc_auc_attack':float(roc_auc_score(y[val_idx],validation_proba)),
 'confusion_matrix_normal_attack':[[tn,fp],[fn,tp]],
 'tn':tn,'fp':fp,'fn':fn,'tp':tp
}
print('VALIDATION only — no test evaluation')
print(json.dumps(metrics,indent=2))
print(classification_report(y[val_idx],validation_pred,labels=[0,1],target_names=['Normal','Attack'],zero_division=0))

In [ ]:
# Model output is local, NOT trusted input from elsewhere. Loading third-party joblib files can execute code.
model_path=MODEL_DIR/'edge_iiotset_static_dt_baseline_v1.joblib'
report_path=REPORT_DIR/'edge_iiotset_static_dt_validation_v1.json'

report={
 'experiment':'Edge-IIoTset provisional static numeric metadata Decision Tree baseline',
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'raw_csv_filename':CSV.name,'raw_sha256':actual_hash,'total_source_rows':len(df),
 'features':FEATURES,'label_binary':'Attack_label','label_attack_type_metadata':'Attack_type',
 'split_strategy':'candidate GroupShuffleSplit by identical selected input-feature vectors, 70/15/15 group target, selected by class coverage and ratio only',
 'split_attempt':attempt,'split_summary':split_summary,'split_manifest':str(manifest_path),
 'validation_metrics':metrics,
 'limitations':[
   'No original capture/session provenance verified; not an independent scenario split',
   'Provisional static binary intrusion classifier; not temporal-window or APT-campaign detection',
   'Feature definitions/live Raspberry Pi extractor parity unverified',
   'No final test labels used in evaluation; evaluate once after frozen model selection',
 ]
}
report_path.write_text(json.dumps(report,indent=2),encoding='utf-8')
joblib.dump(pipe,model_path)
print('Saved local pipeline:',model_path)
print('Saved validation-only report:',report_path)

## What to send back

1. The *feature audit* table and list of selected features.
2. Duplicate input-feature group counts and label-conflict counts.
3. Achieved train/validation/test **row** ratios and attack-type coverage.
4. Validation confusion matrix, attack recall, precision, F1, false-positive rate and PR-AUC.
5. Any `RuntimeError`, especially if the group-disjoint split cannot support the intended 70/15/15 ratio.

**Do not delete the raw CSV or overwrite existing results.** Keep the test set untouched while comparing candidate models. Next, we can train Random Forest, Extra Trees, LightGBM and AdaBoost on the **same frozen train/validation partitions**, then select a small ensemble. Separately, obtain verified PCAP/testbed timestamps for genuine 5/10/15/20-second experiments.